# SRKR 2D Notebook

Runs a two-dimensional spatial scan. Valid fast/slow combinations are `(x, y)` and `(y, x)`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src" / "kohdalab").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/kohdalab.")


repo_root = find_repo_root()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
config_path = repo_root / "src" / "kohdalab" / "config" / "trkr_config_kikuchi.json"

from kohdalab.api import Experiment, format_point, load_config, make_srkr_2d_live_update, srkr_2d_plan

config = load_config(config_path)
experiment = Experiment(config)
zero = config["measurements"]["move_abs"]["zero"]

def print_status(status: str) -> None:
    print(f"status: {status}", flush=True)


In [ ]:
fast_axis = "x"
slow_axis = "y"
ranges = {
    "x": {"min": -2.0, "max": 2.0, "step": 2.0},
    "y": {"min": -2.0, "max": 2.0, "step": 2.0},
}
wait_s = 0.0
return_to_zero = {"fast_axis": True, "slow_axis": True}
output = None  # None uses the output path configured for srkr_2d.
y_key = "R_V"

plan = srkr_2d_plan(
    fast_axis=fast_axis,
    slow_axis=slow_axis,
    ranges=ranges,
    zero_by_axis={
        "t_ps": float(zero.get("t_ps", 0.0)),
        "x_um": float(zero.get("x_um", 0.0)),
        "y_um": float(zero.get("y_um", 0.0)),
    },
    return_to_zero=return_to_zero,
)
axis_key = f"{plan.fast_axis}_cor_um"
live_update = make_srkr_2d_live_update(fast_axis=plan.fast_axis, y_key=y_key, title=plan.summary)


In [ ]:
def handle_point(point):
    print(format_point(point, axis_key=axis_key), flush=True)
    live_update(point)


rows = experiment.run_srkr_2d(
    plan=plan,
    wait_s=wait_s,
    output=output,
    on_status=print_status,
    on_point=handle_point,
)

display(pd.DataFrame(rows))


In [ ]:
# Run this when you are done with the notebook session.
# experiment.disconnect_all()
